# 11d — Decomposição mecanística v2.4: CS-DR × canais primários × 4 specs (B4.M.4)

**Pré-registro v2.4 §7.7 (configuração E) + §6.5 (critério M paramétrico)**

**Mudanças v2.4 vs v2.3.9:** consolidação `cana_direto = res_cana + org_cana` na camada de inferência (Pearson(asinh) = 0,9996, zero células disjuntas — ver §3.10.2 v2.4). 4 outcomes primários em vez de 5. `res_cana` e `org_cana` separados rodados sob FULL para apêndice de cobertura.

**Espelha exatamente** a chamada CS-DR canônica do `11a_estimacao_t1_v4`. Únicas diferenças vs 11a: outcomes da decomposição (não macro-canais) e consolidação cana_direto.

**Opção B**: sub-canais entram via left-join no `panel_canavieiro_main` (842), painel canônico intacto.

**3 bugs do 11a respeitados:** (1) never-treated=NaN em `g_m_cs`; (2) `n_jobs=1`; (3) 6 covs colidentes dropadas.

**Outcomes primários (§3.10.2 v2.4):**
- `asinh_cana_direto` ← consolidado (H5.1)
- `log1p_fert_n`, `log1p_calagem` ← H5.2
- `log1p_res_outros` ← H5.3

**Outcomes de verificação (apêndice cobertura, só FULL):**
- `asinh_res_cana`, `asinh_org_cana` ← componentes de cana_direto, reportados separadamente

**Pré-condições no Drive:**
- `pipeline/b4m4_decomposicao.py` (versão v2.4 — substitua a anterior)
- `data/interim/seeg_subcanais_panel.csv` (B4.M.1, intacto)
- `data/interim/panel_canavieiro_main.csv`
- `data/raw/psm_baseline/base_psm_integrada_raw.csv`
- `outputs_pre/seeg_subcanais_shares_definitivos_v239.csv`

## Setup

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path
BASE_DIR = Path("/content/drive/MyDrive/Renovabio - EcoEco")
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

Mounted at /content/drive


In [2]:
!pip install differences

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 978.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 3.2 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from differences import ATTgt
from pipeline.config import interim, out_pre
from pipeline import b4m4_decomposicao as b4
import importlib; importlib.reload(b4)

# Bootstrap: 199 dry-run, 999 produca0 (pre-registro 7.7)
N_BOOT = 999        # ALTERAR PARA 999 apos dry-run passar
RANDOM_STATE = 42
print(f"setup OK | N_BOOT={N_BOOT} seed={RANDOM_STATE}")
print(f"v2.4 outcomes primarios: {b4.OUTCOMES_PRIMARIOS}")
print(f"v2.4 outcomes verificacao: {b4.OUTCOMES_VERIFICACAO}")

setup OK | N_BOOT=999 seed=42
v2.4 outcomes primarios: ['asinh_cana_direto', 'log1p_fert_n', 'log1p_calagem', 'log1p_res_outros']
v2.4 outcomes verificacao: ['asinh_res_cana', 'asinh_org_cana']


## Bloco 1 — Carregar painel canônico (idêntico ao 11a célula 5)

`geocode` como str, dropar 6 covs colidentes (Bug 3).

In [4]:
panel = pd.read_csv(interim("panel_canavieiro_main.csv"), dtype={"geocode": str})
print(f"panel original: {panel.shape}")

COVS_COLIDENTES = ["gini", "densidade_pop", "log_pop", "idhm_renda",
                   "ivs_capital_humano", "ivs_renda_trabalho"]
drop_cols = [c for c in COVS_COLIDENTES if c in panel.columns]
panel = panel.drop(columns=drop_cols)
print(f"apos dropar {len(drop_cols)} colidentes: {panel.shape}")

assert panel["geocode"].nunique() == 842, panel["geocode"].nunique()
assert sorted(panel["ano"].unique()) == list(range(2015, 2025))
print("OK validacoes (842 munis, 2015-2024)")

panel original: (8420, 139)
apos dropar 6 colidentes: (8420, 133)
OK validacoes (842 munis, 2015-2024)


## Bloco 2 — Join dos sub-canais + consolidação cana_direto (v2.4)

Left-join de `seeg_subcanais_panel.csv` por (geocode, ano). A versão v2.4 do `join_subchannels()` constrói automaticamente `cana_direto = res_cana + org_cana` e `asinh_cana_direto` via `build_cana_direto()` chamada internamente.

In [5]:
seeg_sub = pd.read_csv(interim("seeg_subcanais_panel.csv"))
print(f"seeg_subcanais_panel: {seeg_sub.shape}")

panel = b4.join_subchannels(panel, seeg_sub)
print(f"painel apos join + consolidacao: {panel.shape}")

# Sanity: verificar que cana_direto foi construído
assert "asinh_cana_direto" in panel.columns, "cana_direto AUSENTE - rever modulo v2.4"
print("\nOutcomes primarios:")
for o in b4.OUTCOMES_PRIMARIOS:
    if o in panel.columns:
        nn = panel[o].notna().sum()
        print(f"  {o:22s} notna={nn}/{len(panel)}")
    else:
        print(f"  {o:22s} *** AUSENTE ***")

print("\nOutcomes verificacao (separados, para apendice):")
for o in b4.OUTCOMES_VERIFICACAO:
    if o in panel.columns:
        nn = panel[o].notna().sum()
        print(f"  {o:22s} notna={nn}/{len(panel)}")

# Sanity da consolidacao: cana_direto deve correlacionar ~0.99+ com seus componentes
if "asinh_res_cana" in panel.columns and "asinh_org_cana" in panel.columns:
    corr_rc = panel["asinh_cana_direto"].corr(panel["asinh_res_cana"])
    corr_oc = panel["asinh_cana_direto"].corr(panel["asinh_org_cana"])
    print(f"\nSanity colinearidade (v2.4 §3.10.2):")
    print(f"  corr(asinh_cana_direto, asinh_res_cana) = {corr_rc:.4f}")
    print(f"  corr(asinh_cana_direto, asinh_org_cana) = {corr_oc:.4f}")
    print("  (esperado ~0.99+; confirma que cana_direto absorve ambos)")

seeg_subcanais_panel: (23630, 19)
  join: 842 munis no painel canônico, 842 com cana_direto computado
painel apos join + consolidacao: (8420, 149)

Outcomes primarios:
  asinh_cana_direto      notna=8420/8420
  log1p_fert_n           notna=8420/8420
  log1p_calagem          notna=8420/8420
  log1p_res_outros       notna=8420/8420

Outcomes verificacao (separados, para apendice):
  asinh_res_cana         notna=8420/8420
  asinh_org_cana         notna=8420/8420

Sanity colinearidade (v2.4 §3.10.2):
  corr(asinh_cana_direto, asinh_res_cana) = 0.9990
  corr(asinh_cana_direto, asinh_org_cana) = 0.9983
  (esperado ~0.99+; confirma que cana_direto absorve ambos)


## Bloco 3 — Reconstruir covariáveis (idêntico ao 11a células 8-10)

In [6]:
psm_raw = pd.read_csv(
    BASE_DIR / "data/raw/psm_baseline/base_psm_integrada_raw.csv",
    low_memory=False,
)
psm_raw["geocode"] = psm_raw["0_cd_ibge"].astype(str).str.zfill(7)

BIOMA_FIXES = {"Amaz\ufffd\ufffdnia": "Amazônia",
               "Mata Atl\ufffd\ufffdntica": "Mata Atlântica"}
if "14_bioma" in psm_raw.columns:
    psm_raw["14_bioma"] = psm_raw["14_bioma"].replace(BIOMA_FIXES)

muni_id = (panel.groupby("geocode", as_index=False)
           .agg(municipio=("municipio", "first"), uf=("uf", "first"),
                is_treated_ever=("is_treated_ever", "first"),
                g_m=("g_m", "first"), bioma=("bioma", "first")))
muni_id["treated"] = muni_id["is_treated_ever"].astype(int)

df_cs = muni_id.merge(psm_raw, on="geocode", how="inner")
print(f"df_cs (cross-section): {df_cs.shape}")

df_cs (cross-section): (842, 172)


In [7]:
def safe_log1p(s, idx):
    s = pd.to_numeric(s, errors="coerce") if s is not None else pd.Series(np.nan, index=idx)
    return np.log1p(s.clip(lower=0))
def safe_div(num, den, idx):
    num = pd.to_numeric(num, errors="coerce") if num is not None else pd.Series(np.nan, index=idx)
    den = pd.to_numeric(den, errors="coerce") if den is not None else pd.Series(np.nan, index=idx)
    return np.where((den.notna()) & (den > 0), num / den, np.nan)
def asn(s, idx):
    return pd.to_numeric(s, errors="coerce") if s is not None else pd.Series(np.nan, index=idx)

def build_covariates_raw(df):
    d = df.copy(); idx = d.index
    d["log_pib_total"]=safe_log1p(d.get("1_pib_total"),idx)
    d["log_pib_pc"]=safe_log1p(d.get("1_pib_percap"),idx)
    d["log_pop"]=safe_log1p(d.get("2_pop_2017_ibge"),idx)
    d["log_area_total"]=safe_log1p(d.get("14_area_total"),idx)
    d["densidade_pop"]=safe_div(d.get("2_pop_2017_ibge"),d.get("14_area_total"),idx)
    d["share_vadc_agro"]=safe_div(d.get("1_vadc_agro"),d.get("1_vadc_bruto"),idx)
    d["share_vadc_ind"]=safe_div(d.get("1_vadc_ind"),d.get("1_vadc_bruto"),idx)
    d["share_vadc_serv"]=safe_div(d.get("1_vadc_serv"),d.get("1_vadc_bruto"),idx)
    d["share_vadc_adm"]=safe_div(d.get("1_vadc_adm"),d.get("1_vadc_bruto"),idx)
    d["share_cana_baseline"]=asn(d.get("3_mb_sharegrp_pre_cana"),idx)
    d["mb_share_soja"]=asn(d.get("3_mb_sharegrp_pre_soja"),idx)
    d["mb_share_pastagem"]=asn(d.get("3_mb_sharegrp_pre_pastagem"),idx)
    d["mb_share_vegetacao_nativa"]=asn(d.get("3_mb_sharegrp_pre_vegetacao_nativa"),idx)
    d["mb_share_urbano"]=asn(d.get("3_mb_sharegrp_pre_urbano_infra"),idx)
    d["log_area_cana"]=safe_log1p(d.get("4_area_colhida_ha_cana"),idx)
    d["log_area_soja"]=safe_log1p(d.get("4_area_colhida_ha_soja"),idx)
    d["log_area_agri_total"]=safe_log1p(d.get("4_area_colhida_ha"),idx)
    d["share_area_cana_agri"]=safe_div(d.get("4_area_colhida_ha_cana"),d.get("4_area_colhida_ha"),idx)
    d["share_est_af"]=safe_div(d.get("5_num_est_af"),d.get("5_num_est_total"),idx)
    d["share_est_mp"]=safe_div(d.get("5_num_est_mp"),d.get("5_num_est_total"),idx)
    d["share_area_af"]=safe_div(d.get("6_area_lav_af"),d.get("6_area_lav_total"),idx)
    d["share_area_mp"]=safe_div(d.get("6_area_lav_mp"),d.get("6_area_lav_total"),idx)
    d["trator_per_est"]=safe_div(d.get("11_num_trator_total"),d.get("5_num_est_total"),idx)
    d["share_est_irrig"]=safe_div(d.get("12_num_est_irrig_total"),d.get("5_num_est_total"),idx)
    d["share_area_irrig"]=safe_div(d.get("12_area_irrig_total"),d.get("6_area_lav_total"),idx)
    d["share_est_fin_total"]=safe_div(d.get("13_num_est_fin_total"),d.get("5_num_est_total"),idx)
    d["share_est_at"]=safe_div(d.get("10_num_est_receb_at"),d.get("5_num_est_total"),idx)
    d["natveg_share_area"]=safe_div(d.get("14_vegetacao_natural"),d.get("14_area_total"),idx)
    d["desmat_share_area"]=safe_div(d.get("14_desmatado"),d.get("14_area_total"),idx)
    d["idhm_renda"]=asn(d.get("17_idhm_renda"),idx); d["idhm_educ"]=asn(d.get("17_idhm_educ"),idx)
    d["ivs_infra"]=asn(d.get("17_ivs_infraestrutura_urbana"),idx); d["gini"]=asn(d.get("17_i_gini"),idx)
    d["idhm_long"]=asn(d.get("17_idhm_long"),idx); d["ivs_capital_humano"]=asn(d.get("17_ivs_capital_humano"),idx)
    d["ivs_renda_trabalho"]=asn(d.get("17_ivs_renda_e_trabalho"),idx)
    d["share_est_at_coop"]=safe_div(d.get("10_num_est_receb_at_coop"),d.get("5_num_est_total"),idx)
    d["share_est_at_gov"]=safe_div(d.get("10_num_est_receb_at_gov"),d.get("5_num_est_total"),idx)
    d["share_fin_invest"]=safe_div(d.get("13_num_est_fin_invest"),d.get("5_num_est_total"),idx)
    d["share_fin_cust"]=safe_div(d.get("13_num_est_fin_cust"),d.get("5_num_est_total"),idx)
    d["share_est_trator"]=safe_div(d.get("11_num_est_trator_total"),d.get("5_num_est_total"),idx)
    d["share_est_irrig_pivo"]=safe_div(d.get("12_num_est_irrig_pivo"),d.get("5_num_est_total"),idx)
    d["pct_est_energia"]=asn(d.get("7_est_com_energia%"),idx)
    d["log_area_milho"]=safe_log1p(d.get("4_area_colhida_ha_milho"),idx)
    d["log_area_alg"]=safe_log1p(d.get("4_area_colhida_ha_alg"),idx)
    d["log_area_cafarab"]=safe_log1p(d.get("4_area_colhida_ha_cafarab"),idx)
    d["log_area_cafcan"]=safe_log1p(d.get("4_area_colhida_ha_cafcan"),idx)
    d["mb_share_cafe"]=asn(d.get("3_mb_sharegrp_pre_cafe"),idx)
    d["mb_share_algodao"]=asn(d.get("3_mb_sharegrp_pre_algodao"),idx)
    d["mb_share_silvicultura"]=asn(d.get("3_mb_sharegrp_pre_silvicultura"),idx)
    d["mb_share_agua"]=asn(d.get("3_mb_sharegrp_pre_agua"),idx)
    d["mb_share_outros"]=asn(d.get("3_mb_sharegrp_pre_outros"),idx)
    d["mb_share_agri_total"]=asn(d.get("3_mb_sharegrp_pre_agricultura_total"),idx)
    d["share_num_est_mp"]=safe_div(d.get("5_num_est_mp"),d.get("5_num_est_total"),idx)
    d["share_est_pec"]=safe_div(d.get("5_num_est_pec_total"),d.get("5_num_est_total"),idx)
    d["share_est_lavperm"]=safe_div(d.get("5_num_est_lavperm_total"),d.get("5_num_est_total"),idx)
    d["share_est_lavtemp"]=safe_div(d.get("5_num_est_lavtemp_total"),d.get("5_num_est_total"),idx)
    d["share_area_lavperm"]=safe_div(d.get("6_area_lavperm_total"),d.get("6_area_lav_total"),idx)
    d["share_area_lavtemp"]=safe_div(d.get("6_area_lavtemp_total"),d.get("6_area_lav_total"),idx)
    d["share_area_pec"]=safe_div(d.get("6_area_pec_total"),d.get("6_area_lav_total"),idx)
    d["share_fin_comer"]=safe_div(d.get("13_num_est_fin_comer"),d.get("5_num_est_total"),idx)
    d["share_est_at_propr"]=safe_div(d.get("10_num_est_receb_at_propr"),d.get("5_num_est_total"),idx)
    d["share_est_at_gov_out"]=safe_div(d.get("10_num_est_receb_at_gov_out"),d.get("5_num_est_total"),idx)
    share_cols = [c for c in d.columns if ("share" in c) or c.startswith("mb_share_")]
    for c in share_cols:
        s = pd.to_numeric(d[c], errors="coerce")
        if not s.dropna().empty and (s.dropna().between(-0.05,1.05).mean() > 0.8):
            d[c] = s.clip(0,1)
    return d

df_cs = build_covariates_raw(df_cs)
print("OK covariaveis reconstruidas")

OK covariaveis reconstruidas


In [8]:
COVS_LEAN = ["log_pib_total","log_pib_pc","log_pop","densidade_pop",
             "share_vadc_agro","share_vadc_ind","share_vadc_serv",
             "share_cana_baseline","mb_share_soja","mb_share_pastagem","mb_share_vegetacao_nativa",
             "log_area_cana","log_area_soja","share_area_cana_agri",
             "share_est_af","share_area_af","trator_per_est","share_est_irrig","share_est_fin_total",
             "ivs_infra","gini"]
COVS_FULL = COVS_LEAN + ["share_vadc_adm","idhm_educ","idhm_renda","idhm_long",
                          "ivs_capital_humano","ivs_renda_trabalho",
                          "share_est_at","share_est_at_coop","share_est_at_gov",
                          "share_fin_invest","share_fin_cust",
                          "share_est_trator","share_est_irrig_pivo","pct_est_energia"]
COVS_FULL2 = [c for c in COVS_FULL if c not in ("share_vadc_agro","share_vadc_ind")]
COVS_RICH_FILTRADAS_OUT = ["mb_share_algodao","log_area_cafcan","mb_share_cafe","mb_share_silvicultura"]
COVS_RICH = COVS_FULL + [c for c in [
    "log_area_milho","log_area_alg","log_area_cafarab","log_area_cafcan",
    "mb_share_cafe","mb_share_algodao","mb_share_silvicultura",
    "mb_share_urbano","mb_share_agua","mb_share_outros","mb_share_agri_total",
    "share_num_est_mp","share_est_pec","share_est_lavperm","share_est_lavtemp",
    "share_area_lavperm","share_area_lavtemp","share_area_pec",
    "share_fin_comer","share_est_at_propr","share_est_at_gov_out",
] if c not in COVS_RICH_FILTRADAS_OUT]
assert len(COVS_RICH) == 52, len(COVS_RICH)

SPECS = {"LEAN": COVS_LEAN, "FULL": COVS_FULL, "FULL2": COVS_FULL2, "RICH": COVS_RICH}
for name, covs in SPECS.items():
    missing = [c for c in covs if c not in df_cs.columns]
    assert not missing, f"{name}: faltam {missing}"
    print(f"OK {name:6s}: {len(covs)} covs")

all_covs = sorted(set(COVS_RICH))
for c in all_covs:
    if df_cs[c].isna().any():
        df_cs[c] = df_cs.groupby("uf")[c].transform(lambda x: x.fillna(x.median()))
        df_cs[c] = df_cs[c].fillna(df_cs[c].median())
assert df_cs[all_covs].isna().sum().sum() == 0
print(f"OK imputacao, {len(all_covs)} covs unicas")

OK LEAN  : 21 covs
OK FULL  : 35 covs
OK FULL2 : 33 covs
OK RICH  : 52 covs
OK imputacao, 52 covs unicas


## Bloco 4 — Construir painel CS (idêntico ao 11a célula 12, Bug 1)

In [9]:
panel_cs = panel.merge(df_cs[["geocode"] + all_covs], on="geocode", how="left")
panel_cs["g_m_cs"] = panel_cs["g_m"]  # Bug 1: NaN = never-treated

n_treated = panel_cs["g_m_cs"].notna().sum() // 10
n_never = panel_cs["g_m_cs"].isna().sum() // 10
print(f"tratados: {n_treated} (esperado 194)")
print(f"nunca-tratados: {n_never} (esperado 648)")
assert n_treated == 194 and n_never == 648
print("OK convencao differences: NaN = never-treated")

tratados: 194 (esperado 194)
nunca-tratados: 648 (esperado 648)
OK convencao differences: NaN = never-treated


## Bloco 5 — CS-DR sobre os 4 outcomes PRIMÁRIOS × 4 specs (v2.4)

Consolidação `cana_direto` em H5.1. 4 × 4 = **16 ATTs primários** (era 20 em v2.3.9).

In [10]:
cs_main = b4.run_csdr_outcomes(
    panel_cs,
    outcomes=b4.OUTCOMES_PRIMARIOS,  # v2.4: 4 outcomes (cana_direto consolidado)
    specs=SPECS,
    n_boot=N_BOOT,
    random_state=RANDOM_STATE,
)
cs_main.to_csv(interim("att_canais_main.csv"), index=False)
print(f"\nOK att_canais_main.csv {cs_main.shape}  (esperado 4x4 = 16 ATTs primarios)")
cs_main


>>> asinh_cana_direto
  LEAN   ATT = +0.2867 (SE=0.1589)  [7.2s]
  FULL   ATT = +0.4554 (SE=0.3143)  [4.6s]
  FULL2  ATT = +0.2100 (SE=0.0904)  [2.4s]
  RICH   ATT = +0.5198 (SE=0.5026)  [9.8s]

>>> log1p_fert_n
  LEAN   ATT = +0.1562 (SE=0.0745)  [3.0s]
  FULL   ATT = +0.2358 (SE=0.1380)  [2.9s]
  FULL2  ATT = +0.1238 (SE=0.0481)  [2.3s]
  RICH   ATT = +0.2414 (SE=0.2051)  [4.0s]

>>> log1p_calagem
  LEAN   ATT = +0.0509 (SE=0.0230)  [6.1s]
  FULL   ATT = +0.0435 (SE=0.0244)  [2.7s]
  FULL2  ATT = +0.0381 (SE=0.0184)  [2.3s]
  RICH   ATT = +0.0141 (SE=0.0244)  [4.2s]

>>> log1p_res_outros
  LEAN   ATT = -0.0414 (SE=0.0284)  [6.5s]
  FULL   ATT = -0.0805 (SE=0.0348)  [2.7s]
  FULL2  ATT = -0.0572 (SE=0.0267)  [2.3s]
  RICH   ATT = -0.0893 (SE=0.0456)  [4.0s]

✓ CS-DR: 16/16 sucessos em 66.9s

OK att_canais_main.csv (16, 8)  (esperado 4x4 = 16 ATTs primarios)


,outcome,spec,estimator,ATT,SE,CI_lo,CI_hi,n_munis
0,asinh_cana_direto,LEAN,CS-DR,0.286661,0.158877,-0.024731,0.598054,842
1,asinh_cana_direto,FULL,CS-DR,0.455350,0.314258,-0.160583,1.071284,842
2,asinh_cana_direto,FULL2,CS-DR,0.209955,0.090426,0.032724,0.387186,842
3,asinh_cana_direto,RICH,CS-DR,0.519815,0.502590,-0.465243,1.504872,842
4,log1p_fert_n,LEAN,CS-DR,0.156180,0.074491,0.010181,0.302179,842
5,log1p_fert_n,FULL,CS-DR,0.235800,0.138024,-0.034722,0.506322,842
6,log1p_fert_n,FULL2,CS-DR,0.123755,0.048108,0.029466,0.218044,842
7,log1p_fert_n,RICH,CS-DR,0.241386,0.205081,-0.160565,0.643337,842
8,log1p_calagem,LEAN,CS-DR,0.050949,0.022984,0.005901,0.095997,842
9,log1p_calagem,FULL,CS-DR,0.043452,0.024383,-0.004338,0.091242,842


## Bloco 5b — Verificação cana-rotulados separados (apêndice cobertura)

Roda `res_cana` e `org_cana` SEPARADOS sob FULL apenas, para o apêndice. **Não entram em H5.1 nem em M_direto** (consolidados em `cana_direto`). Servem para mostrar a colinearidade r=0,9996 com transparência no apêndice.

In [11]:
cs_verif = b4.run_csdr_outcomes(
    panel_cs,
    outcomes=b4.OUTCOMES_VERIFICACAO,  # res_cana, org_cana separados
    specs={"FULL": SPECS["FULL"]},      # so FULL para apendice
    n_boot=N_BOOT,
    random_state=RANDOM_STATE,
)
cs_verif.to_csv(interim("att_canais_verificacao.csv"), index=False)
print(f"\nOK att_canais_verificacao.csv {cs_verif.shape}  (2 outcomes x 1 spec = 2 ATTs)")
print("\nNota: res_cana e org_cana sao componentes de cana_direto.")
print("Pearson(asinh) entre eles = 0.9996 (§3.10.2 v2.4).")
print("Estes ATTs sao para apendice de cobertura, NAO para inferencia primaria.")
print("Devem ser quase identicos entre si (colinearidade r=0.9996).")
cs_verif


>>> asinh_res_cana
  FULL   ATT = +0.4346 (SE=0.2971)  [7.2s]

>>> asinh_org_cana
  FULL   ATT = +0.4282 (SE=0.2902)  [2.9s]

✓ CS-DR: 2/2 sucessos em 10.1s

OK att_canais_verificacao.csv (2, 8)  (2 outcomes x 1 spec = 2 ATTs)

Nota: res_cana e org_cana sao componentes de cana_direto.
Pearson(asinh) entre eles = 0.9996 (§3.10.2 v2.4).
Estes ATTs sao para apendice de cobertura, NAO para inferencia primaria.
Devem ser quase identicos entre si (colinearidade r=0.9996).


,outcome,spec,estimator,ATT,SE,CI_lo,CI_hi,n_munis
0,asinh_res_cana,FULL,CS-DR,0.434572,0.297050,-0.147636,1.016779,842
1,asinh_org_cana,FULL,CS-DR,0.428238,0.290161,-0.140467,0.996942,842


## Bloco 6 — Shares definitivos (§6.5 paramétrica)

In [12]:
anos_main = list(range(2015, 2025))
try:
    sh = pd.read_csv(out_pre("seeg_subcanais_shares_definitivos_v239.csv"))
    shares_csv = dict(zip(sh["sub_canal"], sh["share"]))
    # Adiciona cana_direto consolidado (v2.4)
    if "res_cana" in shares_csv and "org_cana" in shares_csv:
        shares_csv["cana_direto"] = shares_csv["res_cana"] + shares_csv["org_cana"]
    shares = shares_csv
    print("shares lidos do artefato v2.3.9 + cana_direto consolidado (v2.4):")
except Exception:
    print("artefato nao encontrado - recalculando de seeg_subcanais_panel:")
    shares = b4.compute_subchannel_shares(seeg_sub, anos_main)

for k, v in shares.items():
    print(f"  s_{k:12s} = {v:.4f}")

print(f"\ns_cana_direto = s_res_cana + s_org_cana = {shares.get('cana_direto', 'N/A')}")

artefato nao encontrado - recalculando de seeg_subcanais_panel:
  s_res_cana     = 0.0534
  s_org_cana     = 0.0415
  s_fert_n       = 0.3199
  s_calagem      = 0.2429
  s_res_outros   = 0.3250
  s_res_minor    = 0.0174
  s_cana_direto  = 0.0949

s_cana_direto = s_res_cana + s_org_cana = 0.09485300941963495


## Bloco 7 — Critério M_direto/M_proxy → Configuração I/II/III (v2.4)

§6.5 v2.4: `M_direto` consolidado (1 termo, `cana_direto`). `M_proxy` inalterado. Fator 1,5.

In [13]:
cfg = b4.decide_configuration(cs_main, shares, spec_principal="FULL", fator=1.5)

print("="*60)
print("CRITERIO DE DECISAO 6.5 v2.4 (spec principal: FULL)")
print("="*60)
print(f"  M_direto = {cfg['M_direto']:.3f}  (consolidado v2.4)")
print(f"  M_proxy  = {cfg['M_proxy']:.3f}  (inalterado)")
print(f"  razao dir/prx = {cfg['razao_direto_proxy']:.2f} (limiar: 1.5)")
print()
print(f"  H5.1 signif (cana_direto <5%)   : {cfg['H5.1_signif']}")
print(f"  H5.2 signif (fert_n OU calagem) : {cfg['H5.2_signif']}")
print(f"  H5.3 nulo   (res_outros n/sig)  : {cfg['H5.3_nulo']}")
print()
print(f"  ATT_cana_direto = {cfg['ATT_cana_direto']:+.4f} (s={cfg['s_cana_direto']:.4f})")
print(f"  ATT_fert_n      = {cfg['ATT_fert_n']:+.4f} (s={cfg['s_fert_n']:.4f})")
print(f"  ATT_calagem     = {cfg['ATT_calagem']:+.4f} (s={cfg['s_calagem']:.4f})")
print(f"  ATT_res_outros  = {cfg['ATT_res_outros']:+.4f}")
print()
print(f"  >>> CONFIGURACAO {cfg['configuracao']} <<<")
print(f"  {cfg['narrativa']}")

pd.DataFrame([cfg]).to_csv(interim("att_canais_config.csv"), index=False)
print("\nOK att_canais_config.csv salvo")

CRITERIO DE DECISAO 6.5 v2.4 (spec principal: FULL)
  M_direto = 4.801  (consolidado v2.4)
  M_proxy  = 0.916  (inalterado)
  razao dir/prx = 5.24 (limiar: 1.5)

  H5.1 signif (cana_direto <5%)   : False
  H5.2 signif (fert_n OU calagem) : False
  H5.3 nulo   (res_outros n/sig)  : False

  ATT_cana_direto = +0.4554 (s=0.0949)
  ATT_fert_n      = +0.2358 (s=0.3199)
  ATT_calagem     = +0.0435 (s=0.2429)
  ATT_res_outros  = -0.0805

  >>> CONFIGURACAO III <<<
  Efeito disperso — sem canal dominante; ver §6.5 Configuração III

OK att_canais_config.csv salvo


## Bloco 8 — Event-study sob FULL (4 outcomes primários)

v2.4: 4 outcomes em vez de 5 (cana_direto consolidado).

In [14]:
es_rows = []
for outcome in b4.OUTCOMES_PRIMARIOS:    # v2.4: 4 outcomes
    if outcome not in panel_cs.columns:
        continue
    try:
        data_es = (panel_cs.dropna(subset=[outcome])
                   .set_index(["geocode","ano"]).sort_index())
        att_es = ATTgt(data=data_es, cohort_column="g_m_cs")
        att_es.fit(formula=f"{outcome} ~ " + " + ".join(COVS_FULL),
                   est_method="dr", control_group="never_treated",
                   boot_iterations=N_BOOT, random_state=RANDOM_STATE,
                   progress_bar=False, n_jobs=1)
        ev = att_es.aggregate("event")
        ev = ev.reset_index() if isinstance(ev, pd.DataFrame) else pd.DataFrame(ev)
        ev["outcome"] = outcome
        es_rows.append(ev)
        print(f"  OK {outcome}")
    except Exception as e:
        print(f"  XX {outcome}: {type(e).__name__}: {str(e)[:60]}")

if es_rows:
    es_all = pd.concat(es_rows, ignore_index=True)
    es_all.to_csv(interim("att_canais_eventstudy.csv"), index=False)
    print(f"\nOK att_canais_eventstudy.csv {es_all.shape}")

  OK asinh_cana_direto
  OK log1p_fert_n
  OK log1p_calagem
  OK log1p_res_outros

OK att_canais_eventstudy.csv (52, 7)


## Conclusão B4.M.4 v2.4

Se tudo rodou:
- `att_canais_main.csv` — 4 outcomes primários × 4 specs = **16 ATTs**
- `att_canais_verificacao.csv` — res_cana, org_cana separados sob FULL (apêndice cobertura, 2 ATTs)
- `att_canais_config.csv` — M_direto consolidado, M_proxy, Configuração I/II/III
- `att_canais_eventstudy.csv` — event-study FULL × 4 primários

**Dry-run N_BOOT=199:** confira sinais plausíveis. **NÃO interprete a Configuração** — IC instáveis com 199.

**Esperado vs v2.3.9 (com base nos números do dry-run anterior):**
- ATT_cana_direto ≈ +0,43 (similar a ATT_res_cana, por colinearidade)
- M_direto cai de ~18,5 → ~4,6 (eliminação da dupla contagem)
- Razão M_direto/M_proxy cai de ~20 → ~5 (ainda > 1,5, mas honesta)

Se OK, **altere N_BOOT=999** e rode de novo (~30min) para o resultado final.

**Próximo:** B4.M.5 (heterogeneidade share-cana, configuração D) usa `share_cana_eq52_pre2018.csv`.

In [15]:
# ============================================================================
# ANÁLISE PÓS-999 — Significância 5% × spec, robustez, e diagnóstico res_outros
# Cole no 11d ao final, APÓS o N_BOOT=999 ter rodado.
# Reusa cs_main e cs_verif em memória.
# ============================================================================
import numpy as np
import pandas as pd

# Limiar 5% bicaudal
Z_CRIT = 1.959963985

def sig_5pct(att, se):
    """Significante a 5% bicaudal."""
    if not (np.isfinite(att) and np.isfinite(se) and se > 0):
        return False, np.nan
    z = att / se
    return bool(abs(z) > Z_CRIT), z

# ─────────────────────────────────────────────────────────────────────────
# 1. TABELA PRINCIPAL — 4 outcomes × 4 specs com flag de significância 5%
# ─────────────────────────────────────────────────────────────────────────
print("=" * 78)
print("TABELA DE SIGNIFICÂNCIA 5% — 4 outcomes primários × 4 specs (999 boot)")
print("=" * 78)
print(f"{'outcome':<22}{'spec':<7}{'ATT':>10}{'SE':>9}"
      f"{'|z|':>8}{'CI lo':>10}{'CI hi':>10}  sig 5%")
print("-" * 78)

ordem_specs = ["LEAN", "FULL", "FULL2", "RICH"]
ordem_outc  = ["asinh_cana_direto", "log1p_fert_n",
               "log1p_calagem", "log1p_res_outros"]

tabela_sig = []
for outc in ordem_outc:
    for spec in ordem_specs:
        row = cs_main.query("outcome == @outc and spec == @spec").iloc[0]
        sig, z = sig_5pct(row["ATT"], row["SE"])
        flag = "  ✓" if sig else "  ·"
        tabela_sig.append({
            "outcome": outc, "spec": spec,
            "ATT": row["ATT"], "SE": row["SE"], "z": z,
            "CI_lo": row["CI_lo"], "CI_hi": row["CI_hi"],
            "sig_5pct": sig,
        })
        print(f"{outc:<22}{spec:<7}{row['ATT']:+10.4f}{row['SE']:>9.4f}"
              f"{abs(z):>8.2f}{row['CI_lo']:>+10.4f}{row['CI_hi']:>+10.4f}{flag}")
    print()

df_sig = pd.DataFrame(tabela_sig)
df_sig.to_csv(interim("att_canais_significancia_5pct.csv"), index=False)

# ─────────────────────────────────────────────────────────────────────────
# 2. RESUMO POR OUTCOME — em quantas specs é significante?
# ─────────────────────────────────────────────────────────────────────────
print("=" * 78)
print("RESUMO POR OUTCOME — robustez através das specs")
print("=" * 78)
print(f"{'outcome':<22}{'n sig 5%':>10}{'/ 4':<6}{'specs significantes':<35}")
print("-" * 78)
for outc in ordem_outc:
    sub = df_sig.query("outcome == @outc")
    n_sig = int(sub["sig_5pct"].sum())
    specs_sig = sub.query("sig_5pct")["spec"].tolist()
    specs_str = ", ".join(specs_sig) if specs_sig else "—"
    print(f"{outc:<22}{n_sig:>10}{' / 4':<6}{specs_str:<35}")
print()

# ─────────────────────────────────────────────────────────────────────────
# 3. VERIFICAÇÃO (Bloco 5b) — res_cana e org_cana separados (FULL)
# ─────────────────────────────────────────────────────────────────────────
print("=" * 78)
print("BLOCO 5b VERIFICAÇÃO — res_cana e org_cana separados sob FULL")
print("=" * 78)
print(f"{'outcome':<22}{'ATT':>10}{'SE':>9}{'|z|':>8}"
      f"{'CI lo':>10}{'CI hi':>10}  sig 5%")
print("-" * 78)
for outc in ["asinh_res_cana", "asinh_org_cana"]:
    row = cs_verif.query("outcome == @outc and spec == 'FULL'").iloc[0]
    sig, z = sig_5pct(row["ATT"], row["SE"])
    flag = "  ✓" if sig else "  ·"
    print(f"{outc:<22}{row['ATT']:+10.4f}{row['SE']:>9.4f}"
          f"{abs(z):>8.2f}{row['CI_lo']:>+10.4f}{row['CI_hi']:>+10.4f}{flag}")

# Diferença entre os dois (deve ser quase zero pela colinearidade r=0.9996)
rc = cs_verif.query("outcome == 'asinh_res_cana' and spec == 'FULL'").iloc[0]
oc = cs_verif.query("outcome == 'asinh_org_cana' and spec == 'FULL'").iloc[0]
diff_att = rc["ATT"] - oc["ATT"]
print(f"\n  Diferença ATT(res_cana) − ATT(org_cana) = {diff_att:+.4f}")
print(f"  (esperado próximo de zero pela colinearidade r=0,9996 §3.10.2)")

# ─────────────────────────────────────────────────────────────────────────
# 4. DIAGNÓSTICO DE ROBUSTEZ DE SINAL — sinal estável através de specs?
# ─────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 78)
print("ROBUSTEZ DE SINAL (todas 4 specs concordam no sinal?)")
print("=" * 78)
for outc in ordem_outc:
    sub = df_sig.query("outcome == @outc")
    sinais = sub["ATT"].apply(lambda x: "+" if x > 0 else ("-" if x < 0 else "0"))
    n_pos = (sub["ATT"] > 0).sum()
    n_neg = (sub["ATT"] < 0).sum()
    if n_pos == 4:
        veredito = "POSITIVO em todas (robusto)"
    elif n_neg == 4:
        veredito = "NEGATIVO em todas (robusto)"
    else:
        veredito = f"MISTO ({n_pos}+, {n_neg}-)  *** ATENÇÃO ***"
    print(f"  {outc:<22} {' '.join(sinais.tolist()):<10}  {veredito}")

# ─────────────────────────────────────────────────────────────────────────
# 5. ATT_cana_direto vs componentes (sanity da consolidação)
# ─────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 78)
print("SANITY DA CONSOLIDAÇÃO — ATT_cana_direto vs componentes (FULL)")
print("=" * 78)
att_cd = cs_main.query("outcome == 'asinh_cana_direto' and spec == 'FULL'"
                       ).iloc[0]["ATT"]
print(f"  ATT[asinh_cana_direto]       = {att_cd:+.4f}")
print(f"  ATT[asinh_res_cana]          = {rc['ATT']:+.4f}")
print(f"  ATT[asinh_org_cana]          = {oc['ATT']:+.4f}")
print(f"\n  ATT(cana_direto) deve ser próximo dos componentes individuais")
print(f"  (não a soma — asinh não é linear; pela colinearidade ~ qualquer um")
print(f"   dos dois)")

# ─────────────────────────────────────────────────────────────────────────
# 6. DIAGNÓSTICO res_outros<0 (a pergunta interpretativa em aberto)
# ─────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 78)
print("FOCO INTERPRETATIVO — res_outros < 0 (canal não-cana, H5.3)")
print("=" * 78)
ro = df_sig.query("outcome == 'log1p_res_outros'")
print(f"{'spec':<8}{'ATT':>10}{'SE':>9}{'|z|':>8}"
      f"{'CI lo':>10}{'CI hi':>10}  sig 5%")
print("-" * 60)
for _, row in ro.iterrows():
    flag = "  ✓" if row["sig_5pct"] else "  ·"
    print(f"{row['spec']:<8}{row['ATT']:+10.4f}{row['SE']:>9.4f}"
          f"{abs(row['z']):>8.2f}{row['CI_lo']:>+10.4f}"
          f"{row['CI_hi']:>+10.4f}{flag}")

n_neg_sig = int(((ro["ATT"] < 0) & ro["sig_5pct"]).sum())
print(f"\n  → ATT negativo significante a 5% em {n_neg_sig} de 4 specs")
if n_neg_sig >= 3:
    print("  → ROBUSTO: efeito negativo em res_outros é achado consistente.")
    print("    Hipóteses interpretativas a discutir:")
    print("      H5.3a: substituição de área (cana ocupa terra antes em")
    print("             outras culturas → menos resíduos não-cana)")
    print("      H5.3b: spillover de práticas de cultivo (manejo cana")
    print("             contagia outras culturas no mesmo município)")
    print("      H5.3c: artefato de alocação SEEG (verificar via níveis brutos)")
elif n_neg_sig >= 1:
    print("  → PARCIAL: efeito negativo emerge em algumas specs.")
    print("    Pode ser sensível à parametrização — recomenda nota")
    print("    de transparência no paper.")
else:
    print("  → NULO em todas: H5.3 confirmada (controle não-cana sem efeito).")
    print("    O sinal negativo do dry-run 199 era ruído de bootstrap pequeno.")

print("\n" + "=" * 78)
print("Salvo: att_canais_significancia_5pct.csv (para a Tabela 4 do paper)")
print("=" * 78)

TABELA DE SIGNIFICÂNCIA 5% — 4 outcomes primários × 4 specs (999 boot)
outcome               spec          ATT       SE     |z|     CI lo     CI hi  sig 5%
------------------------------------------------------------------------------
asinh_cana_direto     LEAN      +0.2867   0.1589    1.80   -0.0247   +0.5981  ·
asinh_cana_direto     FULL      +0.4554   0.3143    1.45   -0.1606   +1.0713  ·
asinh_cana_direto     FULL2     +0.2100   0.0904    2.32   +0.0327   +0.3872  ✓
asinh_cana_direto     RICH      +0.5198   0.5026    1.03   -0.4652   +1.5049  ·

log1p_fert_n          LEAN      +0.1562   0.0745    2.10   +0.0102   +0.3022  ✓
log1p_fert_n          FULL      +0.2358   0.1380    1.71   -0.0347   +0.5063  ·
log1p_fert_n          FULL2     +0.1238   0.0481    2.57   +0.0295   +0.2180  ✓
log1p_fert_n          RICH      +0.2414   0.2051    1.18   -0.1606   +0.6433  ·

log1p_calagem         LEAN      +0.0509   0.0230    2.22   +0.0059   +0.0960  ✓
log1p_calagem         FULL      +0.0435   0